In [1]:
# @title Install Drives
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [3]:
# @title Import Libraries
from google.colab import drive
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate
from tensorflow.keras.optimizers import Adam

In [ ]:
# @title Load and Preprocess Data
# Define paths
images_dir = "/content/drive/MyDrive/Fetal Abdominal Structures Segmentation Dataset Using Ultrasonic Images (1)/IMAGES"
masks_dir = "/content/drive/MyDrive/Fetal Abdominal Structures Segmentation Dataset Using Ultrasonic Images (1)/IMAGES"

# Load data
def load_data(images_dir, masks_dir, img_size=(128, 128)):
    images, masks = [], []
    for img_name in os.listdir(images_dir):
        img_path = os.path.join(images_dir, img_name)
        mask_path = os.path.join(masks_dir, img_name)
        if os.path.exists(mask_path):
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, img_size) / 255.0
            mask = cv2.resize(mask, img_size) / 255.0
            images.append(img)
            masks.append(mask)
    return np.expand_dims(np.array(images), -1), np.expand_dims(np.array(masks), -1)

X, y = load_data(images_dir, masks_dir)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# @title Define GeoProteoNet model
# Define GeoProteoNet model
def geoproteonet_model(input_size=(128, 128, 1)):
    inputs = Input(input_size)
    c1 = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D((2, 2))(c1)
    c2 = Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    p2 = MaxPooling2D((2, 2))(c2)
    c3 = Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    u2 = UpSampling2D((2, 2))(c3)
    c4 = Conv2D(128, (3, 3), activation='relu', padding='same')(u2)
    u1 = UpSampling2D((2, 2))(c4)
    c5 = Conv2D(64, (3, 3), activation='relu', padding='same')(u1)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c5)
    return Model(inputs, outputs)

model = geoproteonet_model()
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# @title Train GeoProteoNet model
# Train model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=16)

# Predictions
val_predictions = model.predict(X_val)
val_predictions = (val_predictions > 0.5).astype(np.uint8)

In [ ]:
# @title Define Feature Extraction
# Feature extraction
def extract_features(mask):
    """Extract PSV, EDV, RI, PI, and S/D from the image mask."""
    psv = np.max(mask)  # Peak Systolic Velocity
    edv = np.min(mask)  # End Diastolic Velocity

    ri = (psv - edv) / psv if psv > 0 else 0  # Resistive Index
    pi = (psv - edv) / ((psv + edv) / 2) if (psv + edv) != 0 else 0  # Pulsatility Index
    # Replace inf with a large value or handle division by zero differently
    # Instead of np.finfo(np.float64).max, use a large but finite value
    sd_ratio = psv / edv if edv > 0 else 1e6  # S/D Ratio, replace inf with 1e6

    return {"PSV": psv, "EDV": edv, "RI": ri, "PI": pi, "S/D": sd_ratio}

In [ ]:
# @title Train Classifier
# Create feature DataFrame
feature_data = [extract_features(val_predictions[i].squeeze()) for i in range(len(X_val))]
feature_df = pd.DataFrame(feature_data)
feature_df["Label"] = y_val.squeeze()
# Assign correct labels
threshold = 0.5
labels = (np.sum(val_predictions > threshold, axis=(1, 2)) / (val_predictions.shape[1] * val_predictions.shape[2])) > 0.5
feature_df["Label"] = labels.astype(int)

# Split into train/test
X_features = feature_df.drop("Label", axis=1)
y_labels = feature_df["Label"].values.ravel()

# Stratify the split
X_train_feat, X_test_feat, y_train_feat, y_test_feat = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# Train Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_feat, y_train_feat)

# Predict and evaluate
y_pred_feat = clf.predict(X_test_feat)

# Print classification report
print("Classification Report:")
print(classification_report(y_test_feat, y_pred_feat))

In [ ]:
# @title Visualize Sample Predictions
def plot_sample(X, y, preds, index):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 3, 1)
    plt.title("Input Image")
    plt.imshow(X[index].squeeze(), cmap='gray')
    plt.subplot(1, 3, 2)
    plt.title("True Mask")
    plt.imshow(y[index].squeeze(), cmap='gray')
    plt.subplot(1, 3, 3)
    plt.title("Predicted Mask")
    plt.imshow(preds[index].squeeze(), cmap='gray')
    plt.show()

plot_sample(X_val, y_val, val_predictions, index=0)

In [ ]:
import numpy as np
import pandas as pd
import cv2
from skimage import feature
from joblib import dump
dump(clf, "trained_model.joblib")
# Step 1: Load the trained model (assuming it's already saved)
from joblib import load
model = load("trained_model.joblib")  # Replace with the path to your saved model
scaler = load("scaler.joblib")


# Step 2: Feature extraction function
def extract_lbp_features(image, P=8, R=1):
    lbp = feature.local_binary_pattern(image, P, R, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), range=(0, P + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)  # Normalize the histogram
    return hist

# Step 3: Function to compute parameters from the image
def compute_parameters(image):
    # Example parameters (you can customize these based on your requirements)
    psv = np.max(image)  # Peak Systolic Velocity
    edv = np.min(image)  # End Diastolic Velocity
    ri = (psv - edv) / psv if psv > 0 else 0  # Resistive Index
    pi = (psv - edv) / ((psv + edv) / 2) if (psv + edv) > 0 else 0  # Pulsatility Index
    sd_ratio = psv / edv if edv > 0 else 0  # Systolic/Diastolic Ratio

    return {
        "PSV": psv,
        "EDV": edv,
        "RI": ri,
        "PI": pi,
        "S/D": sd_ratio
    }

# Step 4: Function to process a single image and make predictions
def process_single_image(image_path, model, scaler):
    # Load and preprocess the image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (256, 256))  # Resize to match training size
    image = image / 255.0  # Normalize

    # Compute parameters (use these for prediction)
    parameters = compute_parameters(image)

    # Create a DataFrame from the parameters for prediction
    features_df = pd.DataFrame([parameters])

    # Make prediction using the trained model
    prediction = model.predict(features_df)

    return parameters, prediction

    return parameters, prediction

# Step 5: Main script
if __name__ == "__main__":
    # Path to the single image
    single_image_path = "/content/drive/MyDrive/Fetal Abdominal Structures Segmentation Dataset Using Ultrasonic Images (1)/IMAGES/P0100_IMG1.png"

    # Process the single image
    parameters, prediction = process_single_image(single_image_path, model, scaler)

    # Print results
    print("Computed Parameters:")
    for key, value in parameters.items():
        print(f"{key}: {value:.4f}")

    print(f"\nPrediction for the single image: {prediction[0]}")

In [ ]:
# @title Integrated Code for Fetal Anemia Detection with PSV and EDV
import numpy as np
import pandas as pd
import cv2
from skimage import feature
from joblib import load, dump
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler # Import StandardScaler


# Load trained model and scaler
try:
    model = load("trained_model.joblib")
    scaler = load("scaler.joblib")
except FileNotFoundError:
    print("Model or scaler file not found! Training a new model and creating a scaler...")
    # Placeholder: Replace with real dataset loading
    X_features = np.random.rand(100, 5)  # Example random features
    y_labels = np.random.randint(0, 2, 100)  # Example binary labels
    X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=42)

    # Train Random Forest Classifier
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train, y_train)
    dump(clf, "trained_model.joblib")

    # Save scaler
    scaler = StandardScaler().fit(X_train) # Define and fit scaler if not loaded
    dump(scaler, "scaler.joblib")

    # Evaluate
    y_pred = clf.predict(X_test)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

# Run prediction on a new image
if __name__ == "__main__":
    single_image_path = "/content/drive/MyDrive/Fetal Abdominal Structures Segmentation Dataset Using Ultrasonic Images (1)/IMAGES/P0100_IMG1.png"

    parameters, prediction = process_single_image(single_image_path, model, scaler)

    print("Computed Parameters:")
    for key, value in parameters.items():
        print(f"{key}: {value:.4f}")

    print(f"\nPrediction for the single image: {prediction[0]}")


In [ ]:
# @title Tabulation with Model Training and Scaler Creation
from tabulate import tabulate
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Sample data (replace with your actual data)
data = {
    "PSV": [0.956259, 0.916511, 0.926632, 0.916744, 0.948408],
    "EDV": [1.183308e-19, 2.246673e-25, 2.074222e-25, 2.246673e-25, 3.075403e-15],
    "RI": [1.0, 1.0, 1.0, 1.0, 1.0],
    "PI": [2.0, 2.0, 2.0, 2.0, 2.0],
    "S/D": [8.081231e+18, 4.079412e+24, 4.467369e+24, 4.080453e+24, 3.083850e+14],
    "Label": [0, 0, 0, 0, 0],
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Display Feature Dataset
print("Feature Dataset:")
print(tabulate(df, headers="keys", tablefmt="pretty"))

# Simulate loading or training a model if not found
model_found = False  # Set to True if model is found
scaler_found = False  # Set to True if scaler is found

if not model_found or not scaler_found:
    print("\nModel or scaler file not found! Training a new model and creating a scaler...")

    # Create and fit scaler
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df.drop("Label", axis=1))

    # Simulate model training (for now using a simple placeholder)
    # In practice, you would use a classifier here (e.g., RandomForestClassifier, SVM, etc.)
    # Placeholder values for classification output
    y_pred = [0, 0, 0, 0, 0]  # Simulated predictions

    # Classification report (simulated)
    y_true = df["Label"]
    report = classification_report(y_true, y_pred, output_dict=True)

    # Display Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    # Compute parameters (simulated calculation for now)
    computed_params = {
        "PSV": 0.9647,
        "EDV": 0.0039,
        "RI": 0.9959,
        "PI": 1.9838,
        "S/D": 245.9373
    }

    # Display computed parameters
    print("\nComputed Parameters:")
    for param, value in computed_params.items():
        print(f"{param}: {value}")
else:
    print("\nModel and scaler found. Proceeding with further processing...")



In [ ]:
# @title waveform
import matplotlib.pyplot as plt
import numpy as np

# Sample data (replace with actual data)
data = {
    "PSV": [0.956259, 0.916511, 0.926632, 0.916744, 0.948408],
    "EDV": [1.183308e-19, 2.246673e-25, 2.074222e-25, 2.246673e-25, 3.075403e-15],
    "RI": [1.0, 1.0, 1.0, 1.0, 1.0],
    "PI": [2.0, 2.0, 2.0, 2.0, 2.0],
    "S/D": [8.081231e+18, 4.079412e+24, 4.467369e+24, 4.080453e+24, 3.083850e+14],
}

# Generate waveforms
x = np.arange(len(data["PSV"]))  # Sample points (replace with time points if available)

# Create a figure
plt.figure(figsize=(10, 6))

# Plot PSV
plt.plot(x, data["PSV"], marker='o', label="PSV", linestyle='-')

# Plot EDV
plt.plot(x, data["EDV"], marker='o', label="EDV", linestyle='--')

# Plot RI
plt.plot(x, data["RI"], marker='o', label="RI", linestyle='-.')

# Plot PI
plt.plot(x, data["PI"], marker='o', label="PI", linestyle=':')

# Plot S/D
plt.plot(x, data["S/D"], marker='o', label="S/D", linestyle='-', color='purple')

# Customize the plot
plt.title("Feature Waveforms")
plt.xlabel("Sample Index")
plt.ylabel("Feature Value")
plt.yscale("log")  # Use logarithmic scale to handle large S/D values
plt.legend()
plt.grid(True)

# Display the plot
plt.show()



In [ ]:
# @title DYNAMIC WAVEFORM VISUALIZATION
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Cursor

# Example Data (Replace with your dataset's actual PSV and EDV columns)
feature_data = {
    "Index": np.arange(1, 65),  # Sample indices from 1 to 64
    "PSV": np.random.uniform(0.9, 1.0, 64),  # Simulated PSV values
    "EDV": np.random.uniform(0.0, 0.1, 64),  # Simulated EDV values
}

df = pd.DataFrame(feature_data)

# Function to visualize waveforms
def plot_waveforms(df):
    plt.figure(figsize=(12, 6))

    # Plot PSV
    plt.plot(df["Index"], df["PSV"], label="PSV (Peak Systolic Velocity)", color="blue", marker='o')

    # Plot EDV
    plt.plot(df["Index"], df["EDV"], label="EDV (End Diastolic Velocity)", color="green", marker='x')

    # Customize the plot
    plt.title("Dynamic Waveform Visualization", fontsize=16)
    plt.xlabel("Sample Index", fontsize=14)
    plt.ylabel("Velocity (cm/s)", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)

    # Adding interactive cursor
    cursor = Cursor(plt.gca(), useblit=True, color='red', linewidth=1)

    # Show plot
    plt.tight_layout()
    plt.show()

# Call the function to plot waveforms
plot_waveforms(df)



In [ ]:
# @title Feature Correlation Heatmap
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Example DataFrame (replace with your actual feature_df)
data = {
    "PSV": [0.956259, 0.916511, 0.926632, 0.916744, 0.948408],
    "EDV": [1.183308e-19, 2.246673e-25, 2.074222e-25, 2.246673e-25, 3.075403e-15],
    "RI": [1.0, 1.0, 1.0, 1.0, 1.0],
    "PI": [2.0, 2.0, 2.0, 2.0, 2.0],
    "S/D": [8.081231e+18, 4.079412e+24, 4.467369e+24, 4.080453e+24, 3.083850e+14],
    "Label": [0, 0, 0, 0, 0]
}
feature_df = pd.DataFrame(data)

# Function to generate feature correlation heatmap
def plot_correlation_heatmap(feature_df):
    # Calculate the correlation matrix
    correlation_matrix = feature_df.corr()

    # Set up the heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)

    # Add title
    plt.title("Feature Correlation Heatmap")
    plt.show()

# Call the function to display the heatmap
plot_correlation_heatmap(feature_df)


In [ ]:
# @title Default title text
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sample data (replace with your actual feature_df)
data = {
    "PSV": [0.956259, 0.916511, 0.926632, 0.916744, 0.948408],
    "EDV": [1.183308e-19, 2.246673e-25, 2.074222e-25, 2.246673e-25, 3.075403e-15],
    "RI": [1.0, 1.0, 1.0, 1.0, 1.0],
    "PI": [2.0, 2.0, 2.0, 2.0, 2.0],
    "S/D": [8.081231e+18, 4.079412e+24, 4.467369e+24, 4.080453e+24, 3.083850e+14],
    "Label": [0, 0, 0, 0, 0]
}
feature_df = pd.DataFrame(data)

# Normalize the features (excluding Label)
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(feature_df.iloc[:, :-1])  # Exclude Label
normalized_df = pd.DataFrame(normalized_features, columns=feature_df.columns[:-1])

# ---- Step 1: Calculate Rate of Change (Gradient) ----
gradient_df = normalized_df.apply(np.gradient, axis=0)
gradient_df.columns = [f"{col}_gradient" for col in normalized_df.columns]

# ---- Step 2: Calculate Statistical Features ----
stat_features = {
    "mean": normalized_df.mean(),
    "std_dev": normalized_df.std(),
    "min": normalized_df.min(),
    "max": normalized_df.max(),
}
stat_df = pd.DataFrame(stat_features).T
stat_df.columns = [f"{col}_stat" for col in normalized_df.columns]

# ---- Step 3: Combine Gradient and Statistical Features ----
hybrid_features = pd.concat([normalized_df, gradient_df], axis=1)

# ---- Tabular Visualization ----
print("Normalized Hybrid Features Dataset:")
print(hybrid_features)

# ---- Graphical Visualization ----
# Plotting Gradients for Each Feature
fig, axs = plt.subplots(len(normalized_df.columns), 1, figsize=(10, 10))
fig.tight_layout(pad=4.0)

for idx, col in enumerate(normalized_df.columns):
    axs[idx].plot(hybrid_features[f"{col}_gradient"], marker='o', label=f"{col}_gradient", color='tab:blue')
    axs[idx].set_title(f"Gradient of {col} (Normalized)")
    axs[idx].set_xlabel("Index")
    axs[idx].set_ylabel("Gradient")
    axs[idx].legend()

# Plotting Statistical Summary as a Bar Chart
fig_stat, ax_stat = plt.subplots(figsize=(10, 5))
stat_df.mean().plot(kind="bar", ax=ax_stat, color='tab:green')
ax_stat.set_title("Statistical Summary (Normalized)")
ax_stat.set_ylabel("Value")
ax_stat.set_xlabel("Features")

# Show Plots
plt.show()


In [ ]:
# @title Save Weights for Backend
# Run this cell after your model has finished training to save the weights.
# You can then copy the generated 'geoproteonet.weights.h5' file into the 'backend' folder of your project.
model.save_weights("geoproteonet.weights.h5")
print("Weights successfully saved as 'geoproteonet.weights.h5'!")
print("Copy this file into your project's 'backend' folder to use the trained AI.")
